<a href="https://colab.research.google.com/github/taka010116/VOICEROID/blob/main/sbv2_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SBV2学習用のGoogle Colabスクリプト
公式の下記の実装を使わせていただいております。

https://github.com/litagin02/Style-Bert-VITS2/blob/master/colab.ipynb

「ランタイム」→「ランタイムのタイプを変更」

から、「T4 GPU」に変更してから、「Shift+Enter」で上からセルを実行してください。

# Google Driveのマウント
認証が入るので、最初のセルに置くのをオススメします。

In [ ]:
#Google Driveのフォルダをマウント（認証入る）
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 必要パッケージのインストール

In [ ]:
# このセルを実行して環境構築してください。
# エラーダイアログ「WARNING: The following packages were previously imported in this runtime: [pydevd_plugins]」が出るが「キャンセル」を選択して続行してください。

import os

os.environ["PATH"] += ":/root/.cargo/bin"

!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/litagin02/Style-Bert-VITS2.git
%cd Style-Bert-VITS2/
!uv pip install --system -r requirements-colab.txt
!python initialize.py --skip_default_models

downloading uv 0.10.9 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'Style-Bert-VITS2'...
remote: Enumerating objects: 7075, done.
remote: Total 7075 (delta 0), reused 0 (delta 0), pack-reused 7075 (from 1)
Receiving objects: 100% (7075/7075), 16.75 MiB | 21.80 MiB/s, done.
Resolving deltas: 100% (4687/4687), done.
/content/Style-Bert-VITS2
Using Python 3.12.12 environment at: /usr
Resolved 179 packages in 3.00s
Prepared 44 packages in 7.89s
Uninstalled 3 packages in 48ms
Installed 44 packages in 421ms
 + asteroid-filterbanks==0.4.0
 + cmudict==1.1.3
 + cn2an==0.5.23
 + colorlog==6.10.1
 + distance==0.1.3
 + docopt==0.6.2
 + g2p-en==2.1.0
 + hyperpyyaml==1.2.3
 + julius==0.2.7
 - librosa==0.11.0
 + librosa==0.9.2
 + lightning==2.6.1
 + lightning-utilities==0.15.3
 + loguru==0.7.3
 - nltk==3.9.1
 + nltk==3.8.1
 + num2words==0.5.14
 - numpy==2.0.2
 + numpy==1.26.4
 + onnx==1.20.1
 + onnxconverter-common==1.16.

# 各種パスやモデル名の指定

In [ ]:
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/val.list

rm: cannot remove '/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list': No such file or directory
rm: cannot remove '/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/val.list': No such file or directory


In [ ]:
#!ls /content/drive/MyDrive
#!ls /content/drive/MyDrive/colab_SBV2train_sample

model_name = "amitaro"

# Google Driveでのパスを特定する
import glob
import os
drive_path = os.path.dirname(glob.glob('/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/SBV2-train.ipynb', recursive=True)[0])
print(drive_path)

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = f"{drive_path}/Data"

# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = f"{drive_path}/model_assets"

#モデルの保存場所が作成されていなければ作成する
import os
os.makedirs(assets_root, exist_ok=True)

import yaml
with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample


# （オプション）ユーザ辞書を指定する

SBV2では、特殊な読み方をする単語などに対して、ユーザ側が読みを指定できる機能がある。

例えば「Aivis」をそのままSBV2に読ませると「エーアイヴイアイエス」と読む。一方で、適切にユーザ辞書ファイル「default.csv」を更新すると「アイヴィス」と読ませることができる。

学習時に、ユーザ辞書を適用させる場合は、Google Driveのユーザ辞書を更新後、`dir_update_FLAG = True`に変更し、下記のセルを実行する。

すると、Google Driveのユーザ辞書csvファイルが、適切な場所にコピーされるため、学習時に反映される


In [ ]:
#ユーザ辞書を更新しない場合は、このセルを実行しなくて良い。
from pathlib import Path
import shutil

#ユーザ辞書を更新する場合は、下記のフラグをTrueにする。
dir_update_FLAG = False
default_dict_path = f"{drive_path}/dict_data/default.csv"
copy_dir = f"dict_data"

if os.path.exists(default_dict_path) and dir_update_FLAG:
    shutil.copy(default_dict_path, copy_dir)
    print(f"'{default_dict_path}' を '{copy_dir}' にコピーしました。")


# 学習パラメータを設定する

In [ ]:
#実行前に辞書を追加する

# 日本語を学習させる場合は、Trueで良い。他言語を学習させるならFalseにする
use_jp_extra = True

# 基本は4でいい。GPUのメモリに余裕がある場合は、増やすと性能が上がるかも
batch_size = 4

# 30-100程度で十分だと思うが、増やすと性能が高いモデルが存在する確率が上がる
# 今回は学習時間の短縮のために20epochに設定する。
epochs = 20

# 何stepごとにモデルの途中経過を保存するかを指定する。無料のGoogle Driveを利用している場合は、保存容量が少ないため、多めに指定すると良い
# 少なく指定すればするほど、性能の高いモデルが存在する確率が上がり、かつ、不慮の事態で途中で学習が停止した際に、再開することができる
# Google Driveの容量が潤沢にある場合は、200-500程度でも良い
save_every_steps = 1500

#音量の正規化を行うかどうか。基本はFalseで良い
normalize = False

#音声の前後の無音期間をトリミングするかどうか。基本はFalseで良い
trim = False

#読みのエラーが発生した時にどう処理するか。基本はskipで良い
yomi_error = "skip"

In [ ]:
import re

path = "/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path, "r", encoding="utf-8") as f:
    text = f.read()

# カンマ系すべて除去
text = re.sub(r"[，､,]", "、", text)

with open(path, "w", encoding="utf-8") as f:
    f.write(text)

print("全カンマ修正完了")

全カンマ修正完了


In [ ]:
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/val.list
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list.cleaned

In [ ]:
import re

path="/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list"

with open(path,"r",encoding="utf-8") as f:
    text=f.read()

text=text.replace(",", "、")
text=text.replace("?", "？")
text=text.replace(".", "。")

with open(path,"w",encoding="utf-8") as f:
    f.write(text)

print("train.list修正完了")

train.list修正完了


In [ ]:
path="/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path,"r",encoding="utf-8") as f:
    text=f.read()

text=text.replace("?", "？")
text=text.replace("LINE", "ライン")

with open(path,"w",encoding="utf-8") as f:
    f.write(text)

print("修正完了")

修正完了


In [ ]:
import re

path="/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path,"r",encoding="utf-8") as f:
    lines=f.readlines()

new_lines=[]

for line in lines:
    parts=line.split("|")
    text=parts[3]

    text=text.replace("？","。")
    text=text.replace("?","。")
    text=text.replace("…","。")
    text=text.replace("ー","")

    parts[3]=text
    new_lines.append("|".join(parts))

with open(path,"w",encoding="utf-8") as f:
    f.writelines(new_lines)

print("クリーニング完了")

クリーニング完了


In [ ]:
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list
!rm /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/val.list

In [ ]:
path = "/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path, "r", encoding="utf-8") as f:
    for i in range(20):
        print(f.readline())

conversation/2024-04-28_対談あみたろのみ-0.wav|amitaro|JP|声、聞こえてますか、皆さん?

conversation/2024-04-28_対談あみたろのみ-1.wav|amitaro|JP|キャロとお布団さんの声、両方聞こえてますでしょうか?

conversation/2024-04-28_対談あみたろのみ-10.wav|amitaro|JP|えっと、めっちゃ簡単に自己紹介します。えっとー、1997年から続く

conversation/2024-04-28_対談あみたろのみ-100.wav|amitaro|JP|まあ、結構あの、本当に傷ついたというか、こ、こんなの親に見せられないみたいな、隠さなきゃって

conversation/2024-04-28_対談あみたろのみ-101.wav|amitaro|JP|かな?今はそういうのないけど、やっぱそれが結構……な、生身の人間だからさぁ、い、あのー、嫌だったかな?うふふふっ

conversation/2024-04-28_対談あみたろのみ-102.wav|amitaro|JP|そうですねー。でもなんか、線引きって難しいや。うん。

conversation/2024-04-28_対談あみたろのみ-103.wav|amitaro|JP|うんうんうん、何でも使っていいよってやつね

conversation/2024-04-28_対談あみたろのみ-104.wav|amitaro|JP|そう、せっかく頑張って作ったのに……

conversation/2024-04-28_対談あみたろのみ-105.wav|amitaro|JP|うん、そうそうそうそうそうそう、うん。

conversation/2024-04-28_対談あみたろのみ-106.wav|amitaro|JP|そうそうそう、うん。うんうんうん。

conversation/2024-04-28_対談あみたろのみ-107.wav|amitaro|JP|あ、なるほどー、そうなんだ

conversation/2024-04-28_対談あみたろのみ-108.wav|amitaro|JP|ねーさん、本当に……待って、えっと

conversation/2024-04-28_対談あみたろのみ-109.wav|amitaro|JP

# 前処理を実施する

この段階で、テキストベースの書き起こし内容を、音素、アクセント表示に変更する処理や、Bert特徴量などを取得して保存する処理などが実行される。

ユーザ辞書の設定なども、この処理で利用される。
学習を途中から再開する場合は、このセルを実行する必要はない。

下記のエラーが出たら、ランタイムを再起動して、再度最初から実行する。
```
NameError: name '_C' is not defined
```

In [ ]:
!rm -f /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/train.list
!rm -f /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/val.list
!rm -f /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list.cleaned
!rm -f /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/wavs/**/*.bert.pt

In [ ]:
!ls /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/wavs | head

!ls /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/wavs/conversation | head

conversation
whisper
2024-04-28_対談あみたろのみ-0.wav
2024-04-28_対談あみたろのみ-100.wav
2024-04-28_対談あみたろのみ-101.wav
2024-04-28_対談あみたろのみ-102.wav
2024-04-28_対談あみたろのみ-103.wav
2024-04-28_対談あみたろのみ-104.wav
2024-04-28_対談あみたろのみ-105.wav
2024-04-28_対談あみたろのみ-106.wav
2024-04-28_対談あみたろのみ-107.wav
2024-04-28_対談あみたろのみ-108.wav


In [ ]:
path = "/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path,"r",encoding="utf-8") as f:
    lines = f.readlines()

lines = [l.replace("conversation/","") for l in lines]

with open(path,"w",encoding="utf-8") as f:
    f.writelines(lines)

print("fixed")

fixed


In [ ]:
path = "/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path,"r",encoding="utf-8") as f:
    lines = f.readlines()

lines = [l.replace("whisper/","") for l in lines]

with open(path,"w",encoding="utf-8") as f:
    f.writelines(lines)

print("whisper path fixed")


whisper path fixed


In [ ]:
path = "/content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/esd.list"

with open(path,"r",encoding="utf-8") as f:
    lines = f.readlines()

new_lines = []

for l in lines:
    parts = l.strip().split("|")
    if len(parts)==4:
        text = parts[3]
        text = text.replace(",","")
        text = text.replace(".","")
        text = text.replace("、","")
        text = text.replace("。","")
        parts[3] = text
        new_lines.append("|".join(parts)+"\n")

with open(path,"w",encoding="utf-8") as f:
    f.writelines(new_lines)

print("text cleaned")

text cleaned


In [ ]:
#再実行する場合はここは実行しない
#サーバに接続できないエラーが出たら再実行する
#!rm -rf /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/bert


from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker
from style_bert_vits2.nlp.japanese.user_dict import update_dict

!pip uninstall -y torchaudio
!pip install torch==2.1.2 torchaudio==2.1.2

!pip uninstall -y torch torchaudio pyannote.audio
!pip install pyannote.audio==3.1.1


pyopenjtalk_worker.initialize_worker()
update_dict()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=1000,
    yomi_error=yomi_error,
)

Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.1.2
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: pyannote.audio 3.1.1
Uninstalling pyannote.audio-3.1.1:
  Successfully uninstalled pyannote.audio-3.1.1
  Using cached pyannote.audio-3.1.1-py2.py3-none-any.whl.metadata (9.3 kB)
  Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached pyannote.audio-3.1.1-py2.py3-none-any.whl (208 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 817.1 kB/s eta 0:00:00
Using cached torchaudio-2.10.0-cp312-cp312-many

03-13 09:32:17 |  INFO  | train.py:72 | Step 1: start initialization...
model_name: amitaro, batch_size: 4, epochs: 20, save_every_steps: 1500, freeze_ZH_bert: False, freeze_JP_bert: False, freeze_EN_bert: False, freeze_style: False, freeze_decoder: False, use_jp_extra: True
03-13 09:32:17 |WARNING | train.py:103 | Step 1: /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/models already exists, so copy it to backup to /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/models_backup
03-13 09:32:27 |SUCCESS | train.py:132 | Step 1: initialization finished.
03-13 09:32:27 |  INFO  | train.py:137 | Step 2: start resampling...
03-13 09:32:27 |  INFO  | subprocess.py:23 | Running: resample.py -i /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/raw -o /content/drive/MyDrive/colab_AI_sample/colab_SBV2train_sample/Data/amitaro/wavs --num_processes 2 --sr 44100
03-13 09:33:19 |SUCCESS | subprocess.py:38 | Success: resampl

(False,
 'Step 5, Error: スタイル特徴ファイルの生成に失敗しました:\nTraceback (most recent call last):\n  File "/content/Style-Bert-VITS2/style_gen.py", line 8, in <module>\n    from pyannote.audio import Inference, Model\n  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/__init__.py", line 29, in <module>\n    from .core.inference import Inference\n  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/inference.py", line 36, in <module>\n    from pyannote.audio.core.io import AudioFile\n  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py", line 43, in <module>\n    torchaudio.set_audio_backend("soundfile")\n    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\nAttributeError: module \'torchaudio\' has no attribute \'set_audio_backend\'\n')

# 学習に利用する設定ファイルを作成する

学習に利用するパラメータを途中で変更したい場合は、このタイミングで作成される`Data`フォルダのモデル内にある`config.json`を直接編集すれば良い

In [ ]:
# 上でつけたモデル名を入力。学習を途中からする場合はきちんとモデルが保存されているフォルダ名を入力。
#model_name = "your_model_name"

#事前学習重みを変更する場合はこのタイミングでdata/modelの中のsafetensorを変更する

import yaml
from gradio_tabs.train import get_path

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

# 学習を開始する

無料版のGoogle Driveを利用している場合は、15GBしか保存容量がないはずなので、定期的にゴミ箱の中の重みファイルを完全削除してください。

1回途中経過が保存されるたびに、1.5GB程度の容量が必要になります。

In [ ]:
if use_jp_extra:
    !python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}
else:
    !python train_ms.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

2026-03-13 09:14:50.117004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773393290.326932   15405 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773393290.382822   15405 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773393290.807019   15405 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773393290.807059   15405 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773393290.807064   15405 computation_placer.cc:177] computation placer alr

# 実行後Colabノードブックのランタイムを削除する

特に、有料版のColabでは、接続時間に応じてコンピューティングユニットが消費されてしまうので、節約のため、学習完了したら即座にColabのランタイムを削除する必要がある。

下記セルにおいて、`FLAG = True`として、実行すると、ランタイムが接続解除される

In [ ]:
# ノートブックの解放
FLAG = False

from google.colab import runtime

if FLAG:
    runtime.unassign()